# Passive compliance — weight loading

How the hand's **torsional (joint-space) spring stiffness** determines the
steady-state finger deflection under incrementally added weights.  The four
fingers are commanded to hold `WEIGHT_POSE`; each weight step deflects them
further toward extension.  Higher stiffness → smaller angular deviation for
the same load.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('../../'))
from hand_config import (
    STIFFNESS_CONDITIONS,
    WEIGHT_FINGERS, WEIGHT_LABELS, WEIGHT_LOG_DURATION,
    WEIGHT_INDEX, WEIGHT_MIDDLE, WEIGHT_RING, WEIGHT_PINKY,
)

import shutil
if shutil.which('latex') is None:
    plt.rcParams['text.usetex'] = False

COLORS = plt.rcParams['axes.prop_cycle'].by_key()['color']
OUTPUT = os.path.join('outputs', 'weight_compliance_hand')
os.makedirs(OUTPUT, exist_ok=True)

# Motor index of the MCP joint for each finger
FINGER_MCP_IDX = {'index': 5, 'middle': 7, 'ring': 9, 'pinky': 11}

# Per-finger MCP target [rad] — used for deviation computation
FINGER_TARGET = {
    'index':  WEIGHT_INDEX,
    'middle': WEIGHT_MIDDLE,
    'ring':   WEIGHT_RING,
    'pinky':  WEIGHT_PINKY,
}

# Step 0 = no weight; steps 1..len(WEIGHT_LABELS) = soft/medium/hard
STEP_LABELS = {0: 'none'} | {i+1: lbl for i, lbl in enumerate(WEIGHT_LABELS)}

def _synth(k):
    """Synthetic demo data: deflection increases with step and scales with 1/K."""
    rng  = np.random.default_rng(int(k * 1000))
    rows = []
    t = 0.0; dt = 0.02
    n = int(WEIGHT_LOG_DURATION / dt)
    n_steps = len(WEIGHT_LABELS)
    for step in range(n_steps + 1):
        delta = (step / n_steps) * (0.3 / k)
        for _ in range(n):
            q = np.zeros(13)
            for fname, midx in FINGER_MCP_IDX.items():
                tgt = FINGER_TARGET[fname]
                q[midx]   = max(0.0, tgt[0] - delta + rng.normal(0, 0.004))
                q[midx+1] = max(0.0, tgt[1] - delta * 0.6 + rng.normal(0, 0.004))
            row = {'time_s': f'{t:.3f}', 'k_rot': k, 'weight_step': step}
            row.update({f'q_{j}': f'{q[j]:.5f}' for j in range(13)})
            row.update({f'qdot_{j}': '0' for j in range(13)})
            row.update({f'tau_{j}': '0' for j in range(13)})
            rows.append(row); t += dt
    return pd.DataFrame(rows)

DEMO = False
def load(k):
    global DEMO
    path = os.path.join(OUTPUT, f'data_K{k:.3f}.csv')
    if os.path.isfile(path):
        return pd.read_csv(path)
    DEMO = True
    return _synth(k)

DATA = {k: load(k) for k in STIFFNESS_CONDITIONS}
banner = ' DEMO (synthetic) data ' if DEMO else ' measured data '
print(f'Loaded{banner}|  K levels: {STIFFNESS_CONDITIONS}')

## Deviation from target vs weight step

Mean MCP angular deviation ($\theta_\mathrm{target} - \theta_\mathrm{actual}$, in degrees)
as a function of weight step (0 = no weight), averaged across the four fingers.
Error bars show the standard deviation across the logging window.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for i, k in enumerate(STIFFNESS_CONDITIONS):
    d   = DATA[k].copy()
    col = COLORS[i % len(COLORS)]
    steps = sorted(d['weight_step'].unique())
    means, stds = [], []

    for s in steps:
        rows = d[d['weight_step'] == s]
        deviations = []
        for fname, midx in FINGER_MCP_IDX.items():
            q_mcp = rows[f'q_{midx}'].astype(float)
            dev   = np.rad2deg(WEIGHT_POSE[0]) - np.rad2deg(q_mcp)
            deviations.append(dev.values)
        dev_all = np.concatenate(deviations)
        means.append(dev_all.mean()); stds.append(dev_all.std())

    steps, means, stds = np.array(steps), np.array(means), np.array(stds)
    ax.plot(steps, means, 'o-', color=col, lw=2.0,
            label=rf'$K = {k:.2f}$ N$\cdot$m/rad')
    ax.fill_between(steps, means - stds, means + stds, color=col, alpha=0.15)

ax.set_xticks(list(STEP_LABELS.keys()))
ax.set_xticklabels(list(STEP_LABELS.values()))
ax.set_xlabel('Weight condition')
ax.set_ylabel(r'MCP deviation from target [deg]')
ax.set_ylim(bottom=0)
ax.legend(loc='upper left', title='torsional stiffness')
ax.grid(True, lw=0.4, alpha=0.5)
if DEMO:
    fig.text(0.5, 0.5, 'DEMO', fontsize=90, color='0.92',
             ha='center', va='center', rotation=30, zorder=0)
fig.savefig(os.path.join(OUTPUT, 'weight_deviation_vs_step.pdf'))
plt.show()

## Per-finger deviation

Same metric broken down by finger.  Differences across fingers reveal any
asymmetry in the finger geometry or motor efficiency.

In [ ]:
fig, axes = plt.subplots(1, len(WEIGHT_FINGERS), figsize=(13, 5), sharey=True)

for ax, fname in zip(axes, WEIGHT_FINGERS):
    midx = FINGER_MCP_IDX[fname]
    for i, k in enumerate(STIFFNESS_CONDITIONS):
        d     = DATA[k].copy()
        col   = COLORS[i % len(COLORS)]
        steps = sorted(d['weight_step'].unique())
        means = []
        for s in steps:
            rows  = d[d['weight_step'] == s]
            q_mcp = rows[f'q_{midx}'].astype(float)
            means.append((np.rad2deg(WEIGHT_POSE[0]) - np.rad2deg(q_mcp)).mean())
        ax.plot(steps, means, 'o-', color=col, lw=1.8,
                label=rf'$K={k:.2f}$')
    ax.set_title(fname)
    ax.set_xticks(list(STEP_LABELS.keys()))
    ax.set_xticklabels(list(STEP_LABELS.values()), fontsize='small')
    ax.set_xlabel('Weight condition')
    ax.grid(True, lw=0.4, alpha=0.5)

axes[0].set_ylabel('MCP deviation [deg]')
axes[0].set_ylim(bottom=0)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right', title='stiffness [N·m/rad]',
           fontsize='small', ncol=1)
if DEMO:
    fig.text(0.5, 0.5, 'DEMO', fontsize=90, color='0.92',
             ha='center', va='center', rotation=30, zorder=0)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT, 'weight_deviation_per_finger.pdf'))
plt.show()